# Calcul de sensibilités

In [1]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme() 
from numpy.random import default_rng
rng = default_rng()
import pandas as pd

## Problème de charge sinistre

On définit la _charge sinistre totale_ (sur une période $T$) par la variable aléatoire positive

$$
    S = \sum_{i = 1}^N X_i
$$
où $N$ est une variable aléatoire à valeurs dans $\mathbf{N}$ représentant le nombre de sinistres sur la période $T$, et pour $i \ge 1$, $X_i$ est une variable aléatoire à valeurs dans $\mathbf{R}_+$ représentant le coût du i-ème sinistre, avec la convention selon laquelle la somme est nulle si $N = 0$. Les $(X_i)_{i \ge 1}$ sont supposées indépendantes et identiquement distribuées, et indépendantes de $N$ (indépendance fréquences - coûts).

Une modélisation classique est de considérer 

- $N$ de loi de Poisson de paramètre $\lambda > 0$, 
- $X_1$ de loi log-normale de paramètres $\mu > 0$, $\sigma^2 > 0$, c'est à dire $X_1 = \exp(G_1)$ avec $G_1 \sim \mathcal{N}(\mu, \sigma^2)$. 

Le but est d'estimer la **probabilité de dépassement** c'est à dire calculer la probabilité que la charge sinistre totale dépasse un seuil $K$:

$$
    p = \mathbf{P}\bigl[S > K\bigr] \quad \text{pour $K$ grand}
$$

Dans la suite on prend $\lambda = 10$, $\mu = 0.1$ et $\sigma = 0.3$ et on considère plusieurs valeurs du seuil $K$.

On utilisera la notation $S^{(\lambda)}$ pour indiquer la dépendance de variable aléatoire $S = \sum_{i=1}^{N} X_i$ en le paramètre $\lambda > 0$ (paramètre de la loi de Poisson sous-jacente). On s'intéresse à la sensibilité de la probabilité $p$ en fonction de lambda c'est à dire 
$$
    \frac{\partial}{\partial \lambda} p(\lambda) = \frac{\partial}{\partial \lambda} \mathbf{P} \bigl[ S^{\lambda} > K \bigl].
$$

## Méthode des différences finies
Implémenter l'estimateur Monte Carlo basé sur les différences finies d'ordre 2
 
$$
    \frac{\partial}{\partial \lambda} p(\lambda) = \frac{p(\lambda + h) - p(\lambda- h)}{2h} + \mathcal{O}(h^2)
$$
Comme vu en cours, il y a plusieurs façon d'implémenter l'estimateur Monte Carlo dans ce cadre biaisé. 

Le **premier estimateur** naïf $J^{(1)}_{n,h}(\lambda)$ est basé sur des réalisations indépendantes de $S^{(\lambda+h)}$ et $S^{(\lambda-h)}$ et n'est pas efficace: la variance explose lorsque $h$ tend vers 0. Ainsi on pose

$$
    J^{(1)}_{n, h}(\lambda) = \frac{1}{2 h n} \bigl( \sum_{k = 1}^n \mathbf{1}_{\{S^{(\lambda+h)}_k > K\}} - \sum_{k = 1}^n  \mathbf{1}_{\{\tilde S^{(\lambda-h)}_k > K\}} \bigr),
$$
où $(S^{(\lambda+h)}_k)_{k \ge 1}$ et $(\tilde S^{(\lambda-h)}_k)_{k \ge 1}$ sont des suites indépendantes de variables aléatoires _i.i.d._.

Le **deuxième estimateur** $J^{(2)}_{n,h}(\lambda)$ utilise des réalisations fortements corrélées de la loi de Poisson au sens suivant: on utilise la même réalisation uniforme $U$ pour constuire deux réalisations $N^{(\lambda+h)}$ et $N^{(\lambda-h)}$ en utilisant la méthode de l'inverse de la fonction de répartition. Dans ce deuxième estimateur, les lois log-normales sont indépendantes. On a donc

$$
    J^{(2)}_{n, h}(\lambda) = \frac{1}{2 h n} \sum_{k = 1}^n \bigl(\mathbf{1}_{\{S^{(\lambda+h)}_k > K\}} - \mathbf{1}_{\{\bar S^{(\lambda-h)}_k > K\}} \bigr),
$$
où pour $k \ge 1$, $S^{(\lambda+h)}_k = \sum_{i = 1}^{G(\lambda+h, U_k)} X_{i,k}$ et $\bar S^{(\lambda-h)}_k = \sum_{i = 1}^{G(\lambda-h, U_k)} \bar X_{i,k}$ avec $G(\lambda, u)$ l'inverse généralisée de la loi de Poisson de paramètre $\lambda$, $(U_k)_{k \ge 1}$ suite _i.i.d._ uniforme sur $[0,1]$ indépendante de $(X_{i,k})_{i\ge1, k\ge 1}$ et $(\bar X_{i,k})_{i\ge1, k\ge 1}$ deux suites (doublement indicées) _i.i.d._ de loi log-normale (de paramètres $\mu$ et $\sigma$ inchangés).

Un **troisième estimateur** $J^{(3)}_{n,h}(\lambda)$ utilise des réalisations fortements corrélées de la loi de Poisson et des variables aléatoires log-normales communes. 

$$
    J^{(3)}_{n, h}(\lambda) = \frac{1}{2 h n} \sum_{k = 1}^n \bigl(\mathbf{1}_{\{S^{(\lambda+h)}_k > K\}} - \mathbf{1}_{\{S^{(\lambda-h)}_k > K\}} \bigr),
$$
où pour $k \ge 1$, $S^{(\lambda+h)}_k = \sum_{i = 1}^{G(\lambda+h, U_k)} X_{i,k}$ et $S^{(\lambda-h)}_k = \sum_{i = 1}^{G(\lambda-h, U_k)} X_{i,k}$ avec $G(\lambda, u)$ l'inverse généralisée de la loi de Poisson de paramètre $\lambda$, $(U_k)_{k \ge 1}$ suite _i.i.d._ uniforme sur $[0,1]$ indépendante de $(X_{i,k})_{i\ge1, k\ge 1}$ une suite (doublement indicée) _i.i.d._ de loi log-normale.

### Question: plusieurs estimateurs des différences finies 

On fixe les paramètres $\lambda = 10$, $\mu = 0.1$, $\sigma = 0.3$ et $K = 20$.
Programmer ces 3 estimateurs pour différentes valeurs de $h$ (par exemple, $h=1$, 0.5, 0.1 et 0.01), et donner le résultat des estimateurs Monte Carlo avec $n = 50\,000$.

Que se passe-t-il lorsque $h$ tend vers 0? Comparez le comportement pour ces 3 estimateurs. Il est très important de bien interpréter ces tableaux de résultats et de conclure qu'il faut utiliser l'estimateur $J^{(3)}_{n, h}(\lambda)$ et en aucun cas l'estimateur $J^{(1)}_{n,h}(\lambda)$.

_Remarque_: on considère ici uniquement l'étude de l'erreur statistique dûe à la méthode de Monte Carlo. On ne considère pas l'erreur de biais qui décroît lorsque $h$ tend vers 0 et qui est peut-être non négligeable pour $h = 1$. Les IC construits ici sont biaisés et on ne peut pas affirmer que la vraie valeur est dans l'IC à 95% (au moins pour les grandes valeurs de $h$). 

In [2]:
def monte_carlo(sample, proba = 0.95):
    mean = np.mean(sample)
    var = np.var(sample, ddof=1)
    alpha = 1 - proba 
    quantile = stats.norm.ppf(1 - alpha/2)  # fonction quantile 
    ci_size = quantile * np.sqrt(var / sample.size)
    return (mean, var, mean - ci_size, mean + ci_size)

In [3]:
def simu_S(size, mu, sigma, lambd): 
    sample_N = rng.poisson(size=size, lam=lambd)
    sample_S = np.empty(size)
    for k, Nk in enumerate(sample_N):
        sample_S[k] = np.sum(rng.lognormal(size=Nk, mean=mu, sigma = sigma))
    return sample_S

In [4]:
lambd, mu, sigma = 10, 0.1, 0.3
K = 20

In [5]:
def J1(n, h): 
    sample_Sph = simu_S(n, mu, sigma, lambd+h)
    sample_Smh = simu_S(n, mu, sigma, lambd-h)
    xph = (sample_Sph > K).astype(int)
    xmh = (sample_Smh > K).astype(int)
    return monte_carlo((xph - xmh)/(2*h))

In [6]:
# on crée une fonction pour ne pas répéter ces lignes de code
def result_estimator(J, n=50000, hs=[1, 0.5, 0.1, 0.01]):
    result = [ J(n, h) for h in hs ]
    df = pd.DataFrame(result, 
                      columns=['mean', 'var', 'lower', 'upper'], 
                      index=hs)
    return df

In [7]:
result_estimator(J1)

,mean,var,lower,upper
1.00,0.01765,0.012734,0.016661,0.018639
0.50,0.01764,0.044490,0.015791,0.019489
0.10,0.01060,1.027908,0.001713,0.019487
0.01,0.04300,108.950330,-0.048491,0.134491


In [8]:
def J2(n, h): 
    sample_U = rng.random(size=n)
    sample_Nph = stats.poisson(mu = lambd+h).ppf(sample_U)
    sample_Nmh = stats.poisson(mu = lambd-h).ppf(sample_U)
    sample_Sph = np.empty(n)
    sample_Smh = np.empty(n)
    for k, (Nph, Nmh) in enumerate(zip(sample_Nph, sample_Nmh)):
        sample_Sph[k] = np.sum(rng.lognormal(size=int(Nph), mean=mu, sigma = sigma))
        sample_Smh[k] = np.sum(rng.lognormal(size=int(Nmh), mean=mu, sigma = sigma))
    xph = (sample_Sph > K).astype(int)
    xmh = (sample_Smh > K).astype(int)
    return monte_carlo((xph - xmh)/(2*h))

In [9]:
result_estimator(J2)

,mean,var,lower,upper
1.00,0.01784,0.008962,0.017010,0.018670
0.50,0.01794,0.023459,0.016597,0.019283
0.10,0.01260,0.459850,0.006656,0.018544
0.01,0.00000,42.800856,-0.057344,0.057344


In [10]:
def J3(n, h): 
    sample_U = rng.random(size=n)
    sample_Nph = stats.poisson(mu = lambd+h).ppf(sample_U).astype(int)
    sample_Nmh = stats.poisson(mu = lambd-h).ppf(sample_U).astype(int)
    sample_Sph = np.empty(n)
    sample_Smh = np.empty(n)
    for k, (Nph, Nmh) in enumerate(zip(sample_Nph, sample_Nmh)):
        max_N = max(Nph, Nmh)
        sample_X = rng.lognormal(size=max_N, mean=mu, sigma = sigma)
        sample_Sph[k] = np.sum(sample_X[:Nph])
        sample_Smh[k] = np.sum(sample_X[:Nmh])
    xph = (sample_Sph > K).astype(int)
    xmh = (sample_Smh > K).astype(int)
    return monte_carlo((xph - xmh)/(2*h))

In [11]:
result_estimator(J3)

,mean,var,lower,upper
1.00,0.01697,0.008197,0.016176,0.017764
0.50,0.01714,0.016847,0.016002,0.018278
0.10,0.01830,0.091167,0.015653,0.020947
0.01,0.01300,0.649844,0.005934,0.020066
